In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
CONUS climate-change summary script (PRCP, TMAX, TMIN, etc.)

Workflow
--------
1. Load daily & monthly NOAA station data.
2. Build monthly DATE and water year (W_YEAR).
3. Restrict monthly data to WY 1926–2025 (Oct 1925 – Oct 2025).
4. Load Thiessen polygons and ecoregions; compute station areas.
5. Aggregate to annual (per station, per water year).
6. For each variable in `variables_to_run` (e.g., PRCP, TMAX, TMIN):
   - Compute full, recent, and penultimate means per station.
   - Compute recent - full and recent - penult deltas.
   - Summarize by ecoregion and for CONUS.
   - Save station-level and summary CSVs.

To add another “temperature-like” annual variable (e.g., TMEAN, GDD),
make sure it exists in `df_annual` and add its name to `variables_to_run`.
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import data_loader as dl


def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "scripts" else cwd


def find_project_root() -> Path:
    base = get_project_root()
    required_raw = base / "data_raw" / "Thiessen_Polygons_100yrs.csv"
    if not required_raw.exists():
        raise FileNotFoundError(f"Required data file not found: {required_raw}")
    return base

# ============================================================
# 0. BASIC HELPERS
# ============================================================

def compute_water_year(date_series: pd.Series) -> pd.Series:
    """
    Compute water year from a pandas Series of datetime objects.
    Water year starts in October.
    Example: Oct 1924 → WY 1925.
    """
    dates = pd.to_datetime(date_series)
    return dates.apply(lambda d: d.year + 1 if d.month >= 10 else d.year)


def clip_spei_spi(df: pd.DataFrame,
                  lower_bound: float = -3.0,
                  upper_bound: float = 3.0) -> pd.DataFrame:
    """
    Clip all columns containing 'spei' or 'spi' (case-insensitive)
    to the specified range.

    Parameters
    ----------
    df : DataFrame
        Input dataframe.
    lower_bound, upper_bound : float
        Clipping bounds.

    Returns
    -------
    DataFrame
        Dataframe with clipped spei/spi columns.
    """
    spei_spi_cols = [
        c for c in df.columns
        if ("spei" in c.lower()) or ("spi" in c.lower())
    ]
    for col in spei_spi_cols:
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    return df


# ============================================================
# 1. PERIOD MEANS PER STATION
# ============================================================

def _window_mean_per_station(df: pd.DataFrame,
                             value_col: str,
                             time_col: str = "W_YEAR",
                             id_col: str = "ID",
                             start_year: int | None = None,
                             end_year: int | None = None,
                             suffix: str = "full") -> pd.DataFrame:
    """
    Helper: compute mean(value_col) per station over a given time window.

    Returns
    -------
    DataFrame with columns [ID, <value_col>_<suffix>_mean]
    """
    mask = pd.Series(True, index=df.index)
    if start_year is not None:
        mask &= df[time_col] >= start_year
    if end_year is not None:
        mask &= df[time_col] <= end_year

    df_win = df.loc[mask]
    mean_col_name = f"{value_col}_{suffix}_mean"

    out = (
        df_win.groupby(id_col)[value_col]
        .mean()
        .reset_index()
        .rename(columns={value_col: mean_col_name})
    )
    return out


def compute_period_means_per_station(
    df: pd.DataFrame,
    value_col: str = "PRCP",
    time_col: str = "W_YEAR",
    id_col: str = "ID",
    full_start: int = 1926,
    full_end: int = 2025,
    recent_start: int = 1996,
    recent_end: int = 2025,
    penult_start: int | None = None,
    penult_end: int | None = None,
) -> pd.DataFrame:
    """
    Compute:
      - full/centennial mean over [full_start, full_end]
      - recent mean over [recent_start, recent_end]
      - penultimate mean over the 30-year window preceding recent
        (or [penult_start, penult_end] if provided)

    Returns
    -------
    DataFrame with per-station columns:
      ID,
      <val>_full_mean, <val>_recent_mean, <val>_penult_mean
    """
    # Derive penultimate window if not given (same length as recent)
    if penult_start is None or penult_end is None:
        penult_end = recent_start - 1
        penult_start = penult_end - (recent_end - recent_start)

    df = df.copy()

    full_df = _window_mean_per_station(
        df, value_col, time_col, id_col,
        start_year=int(full_start),
        end_year=int(full_end),
        suffix="full",
    )

    recent_df = _window_mean_per_station(
        df, value_col, time_col, id_col,
        start_year=int(recent_start),
        end_year=int(recent_end),
        suffix="recent",
    )

    penult_df = _window_mean_per_station(
        df, value_col, time_col, id_col,
        start_year=int(penult_start),
        end_year=int(penult_end),
        suffix="penult",
    )

    # Merge on ID
    out = full_df.merge(recent_df, on=id_col, how="outer")
    out = out.merge(penult_df, on=id_col, how="outer")
    return out


# ============================================================
# 2. DIFFERENCES USING RECENT AS REFERENCE
# ============================================================

def add_change_columns(df_means: pd.DataFrame,
                       value_col: str = "PRCP",
                       id_col: str = "ID") -> pd.DataFrame:
    """
    Given per-station period means, add:
      - <val>_recent_minus_full
      - <val>_recent_minus_penult

    Returns
    -------
    DataFrame with new columns.
    """
    df = df_means.copy()

    full_col = f"{value_col}_full_mean"
    recent_col = f"{value_col}_recent_mean"
    penult_col = f"{value_col}_penult_mean"

    df[f"{value_col}_recent_minus_full"] = df[recent_col] - df[full_col]
    df[f"{value_col}_recent_minus_penult"] = df[recent_col] - df[penult_col]

    return df


# ============================================================
# 3. MERGE WITH ECOREGIONS
# ============================================================

def attach_ecoregions(station_df: pd.DataFrame,
                      domain_df: pd.DataFrame,
                      id_col: str = "ID",
                      domain_col: str = "domainName") -> pd.DataFrame:
    """
    Merge per-station stats with ecoregion/domain information.
    domain_df must contain [ID, domainName].
    """
    return station_df.merge(
        domain_df[[id_col, domain_col]],
        on=id_col,
        how="left",
    )


# ============================================================
# 4. GENERIC SUMMARY (REGION + CONUS)
# ============================================================

def summarize_metrics_by_region_and_conus(
    df_with_domain: pd.DataFrame,
    metrics: List[str],
    domain_col: str = "domainName",
) -> pd.DataFrame:
    """
    Summarize specified metric columns by ecoregion and for CONUS.

    For each region (including CONUS), returns:
      min, max, mean, median for each metric.

    metrics
        List of column names to summarize, e.g.
        ['PRCP_recent_minus_full', 'PRCP_recent_minus_penult'] or
        ['PRCP_full_mean', 'PRCP_recent_mean'].
    """
    df = df_with_domain.copy()

    # --- 1) Region-level summary ---
    agg_dict = {m: ["min", "max", "mean", "median"] for m in metrics}
    region_summary = df.groupby(domain_col).agg(agg_dict)

    region_summary.columns = [
        f"{metric}_{stat}"
        for metric, stat in region_summary.columns
    ]
    region_summary = region_summary.reset_index()

    # --- 2) CONUS-wide summary (all stations) ---
    conus_row = df[metrics].agg(["min", "max", "mean", "median"]).T

    conus_data: dict[str, float] = {}
    for metric in metrics:
        conus_data[f"{metric}_min"] = conus_row.loc[metric, "min"]
        conus_data[f"{metric}_max"] = conus_row.loc[metric, "max"]
        conus_data[f"{metric}_mean"] = conus_row.loc[metric, "mean"]
        conus_data[f"{metric}_median"] = conus_row.loc[metric, "median"]

    conus_df = pd.DataFrame(
        [{domain_col: "CONUS", **conus_data}]
    )

    # --- 3) Combine region + CONUS ---
    summary_all = pd.concat([region_summary, conus_df], ignore_index=True)
    return summary_all


# ============================================================
# 5. HIGH-LEVEL PIPELINE FOR ONE VARIABLE
# ============================================================

def climate_change_pipeline(
    df_annual: pd.DataFrame,
    df_domains: pd.DataFrame,
    value_col: str,
    time_col: str = "W_YEAR",
    id_col: str = "ID",
    full_start: int = 1926,
    full_end: int = 2025,
    recent_start: int = 1996,
    recent_end: int = 2025,
    penult_start: int | None = None,
    penult_end: int | None = None,
    domain_col: str = "domainName",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Full pipeline for one climate variable (PRCP, TMAX, TMIN, etc.):

      1) Compute full, recent, and penultimate means per station.
      2) Compute differences using recent as reference:
           recent - full, recent - penult.
      3) Attach ecoregions.
      4) Summarize:
           a) differences by ecoregion + CONUS
           b) period means (full & recent) by ecoregion + CONUS

    Returns
    -------
    station_stats : DataFrame
        Per-station means and differences + domainName.
    diff_summary : DataFrame
        Ecoregion + CONUS summary of difference metrics.
    mean_summary : DataFrame
        Ecoregion + CONUS summary of full/recent mean metrics.
    """
    means_df = compute_period_means_per_station(
        df=df_annual,
        value_col=value_col,
        time_col=time_col,
        id_col=id_col,
        full_start=full_start,
        full_end=full_end,
        recent_start=recent_start,
        recent_end=recent_end,
        penult_start=penult_start,
        penult_end=penult_end,
    )

    station_stats = add_change_columns(
        means_df,
        value_col=value_col,
        id_col=id_col,
    )

    station_stats = attach_ecoregions(
        station_stats,
        df_domains,
        id_col=id_col,
        domain_col=domain_col,
    )

    # Difference metrics
    diff_cols = [
        f"{value_col}_recent_minus_full",
        f"{value_col}_recent_minus_penult",
    ]
    diff_summary = summarize_metrics_by_region_and_conus(
        station_stats,
        metrics=diff_cols,
        domain_col=domain_col,
    )

    # Mean metrics (full & recent)
    mean_cols = [
        f"{value_col}_full_mean",
        f"{value_col}_recent_mean",
    ]
    mean_summary = summarize_metrics_by_region_and_conus(
        station_stats,
        metrics=mean_cols,
        domain_col=domain_col,
    )

    return station_stats, diff_summary, mean_summary


# ============================================================
# 6. RUN PIPELINE FOR A VARIABLE AND SAVE
# ============================================================

def run_and_save_for_variable(
    value_col: str,
    df_annual: pd.DataFrame,
    df_domains: pd.DataFrame,
    df_neon_area: pd.DataFrame,
    out_dir: Path,
    full_start: int = 1926,
    full_end: int = 2025,
    recent_start: int = 1996,
    recent_end: int = 2025,
) -> None:
    """
    Run climate_change_pipeline for a single variable and save outputs
    (mean summary, diff summary, and station-level stats with area).
    """
    print(f"\n=== Processing variable: {value_col} ===")

    station_stats, diff_summary, mean_summary = climate_change_pipeline(
        df_annual=df_annual,
        df_domains=df_domains,
        value_col=value_col,
        full_start=full_start,
        full_end=full_end,
        recent_start=recent_start,
        recent_end=recent_end,
    )

    # Round summaries (keep station stats unrounded)
    mean_summary_rounded = mean_summary.round(0)
    diff_summary_rounded = diff_summary.round(0)

    var_lower = value_col.lower()

    mean_summary_path = out_dir / f"{var_lower}_mean_summary.csv"
    diff_summary_path = out_dir / f"{var_lower}_diff_summary.csv"
    station_stats_path = out_dir / f"{var_lower}_station_stats.csv"

    mean_summary_rounded.to_csv(mean_summary_path, index=False)
    diff_summary_rounded.to_csv(diff_summary_path, index=False)

    station_stats_with_area = station_stats.merge(
        df_neon_area, on=["ID", "domainName"], how="left"
    )
    station_stats_with_area.to_csv(station_stats_path, index=False)

    print(f"  Saved mean   summary -> {mean_summary_path}")
    print(f"  Saved diff   summary -> {diff_summary_path}")
    print(f"  Saved station stats  -> {station_stats_path}")


# ============================================================
# 7. MAIN SCRIPT
# ============================================================

def main() -> None:
    # -----------------------------
    # Paths & output directory
    # -----------------------------
    base_dir = find_project_root()
    data_raw_dir = base_dir / "data_raw"
    data_intermediate_dir = base_dir / "data_intermediate"
    data_intermediate_dir.mkdir(parents=True, exist_ok=True)

    # -----------------------------
    # Load daily & monthly NOAA data
    
    df_daily = dl.df_daily.copy()
    df_monthly = dl.df_monthly_updated.copy()

    print("Daily data shape:", df_daily.shape)
    print("Monthly data shape:", df_monthly.shape)

    # -----------------------------
    # Build DATE & water year in monthly
    # -----------------------------
    # df_monthly["DATE"] = pd.to_datetime(
    #     df_monthly[["YEAR", "MONTH"]].assign(DAY=1)
    # )
    # df_monthly["W_YEAR"] = compute_water_year(df_monthly["DATE"])

    # Restrict to WY 1926–2025: Oct 1925–sept 2025
    df_monthly = df_monthly[
        (df_monthly["DATE"] >= "1925-10-01")
        & (df_monthly["DATE"] <= "2025-09-30")
    ].copy()

    # -----------------------------
    # Load Thiessen polygons & ecoregions
    # -----------------------------
    df_neon = pd.read_csv(data_raw_dir / "Thiessen_Polygons_100yrs.csv")
    df_neon = df_neon[["ID_1", "Lat", "Lon", "Area_sqkm"]]
    df_neon.columns = ["ID", "Lat", "Lon", "Area"]

    df_id_neon = pd.read_csv(data_raw_dir / "ID_ecoregion.csv")

    df_id_neon_updated = df_id_neon.copy()
    replace_map = {
        "Atlantic Neotropical": "Southeast",
        "Appalachians / Cumberland Plateau": "Appalachians",
        "Southern Rockies / Colorado Plateau": "Colorado Plateau",
    }
    df_id_neon_updated["domainName"] = df_id_neon_updated["domainName"].replace(
        replace_map
    )
    df_id_neon_updated = df_id_neon_updated[["ID", "domainName"]]

    # Station area with domain info
    df_neon_area = df_neon[["ID", "Area"]].merge(
        df_id_neon_updated, on="ID", how="inner"
    )

    # -----------------------------
    # Annual aggregation
    # -----------------------------
    # NOTE: PRCP is summed over water year; TMAX/TMIN are averaged.
    # df_annual = (
    #     df_monthly
    #     .groupby(["ID", "W_YEAR"])
    #     .agg({
    #         "PRCP": "sum",
    #         "TMAX": "mean",
    #         "TMIN": "mean",
    #     })
    #     .reset_index()
    # )
    df_annual = dl.df_yearly.copy()
    df_annual_updated = df_annual[df_annual["W_YEAR"] < 2026].copy()

    # -----------------------------
    # Run pipeline for each variable
    # -----------------------------
    # For other temperature-like variables (e.g., 'TMEAN', 'GDD'),
    # make sure they are in df_annual_updated and add here:
    variables_to_run = ["PRCP", "TMAX", "TMIN","AI","PCI","WSDI","CSDI","SDII"]

    for var in variables_to_run:
        run_and_save_for_variable(
            value_col=var,
            df_annual=df_annual_updated,
            df_domains=df_id_neon_updated,
            df_neon_area=df_neon_area,
            out_dir=data_intermediate_dir,
            full_start=1926,
            full_end=2025,
            recent_start=1996,
            recent_end=2025,
        )

    print("\nAll variables processed. Outputs in:", data_intermediate_dir.resolve())


if __name__ == "__main__":
    main()


Daily data shape: (32151928, 10)
Monthly data shape: (1035600, 15)

=== Processing variable: PRCP ===
  Saved mean   summary -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate\prcp_mean_summary.csv
  Saved diff   summary -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate\prcp_diff_summary.csv
  Saved station stats  -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate\prcp_station_stats.csv

=== Processing variable: TMAX ===
  Saved mean   summary -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate\tmax_mean_summary.csv
  Saved diff   summary -> C:\Users\adeba\OneDrive\Documents\Datascience\Full_climate_project\climate_100yr\climate_100yr_github_ready\data_intermediate\tmax_diff_summ